In [2]:
import pandas as pd

df = pd.read_csv("hangzhou_housing_dataset.csv ")

print("DataFrame Imported")

DataFrame Imported


In [3]:
df.head()

,id,district,area_sqm,bedrooms,living_rooms,bathrooms,floor,total_floors,has_elevator,building_age_years,...,school_district_tier,west_lake_distance_m,green_ratio,parking_ratio,management_fee_yuan_per_sqm,listing_date,listing_duration_days,price_adjustments_count,unit_price_yuan_per_sqm,total_price_wan
0,1,西湖区,112.4,3,1,3,8,18,1,4.0,...,C,24423.0,0.32,NaN,5.73,3/5/2024,419,1.0,53167,597.60
1,2,萧山区,97.8,3,2,3,2,6,1,15.0,...,B,1247.0,0.37,0.64,NaN,1/23/2024,443,1.0,37711,368.81
2,3,临安区,70.4,2,2,2,7,18,1,34.0,...,B,1146.0,0.25,1.44,5.16,10/1/2023,636,NaN,12077,85.02
3,4,西湖区,90.1,3,2,3,4,11,1,3.0,...,C,11585.0,0.38,0.81,4.88,8/5/2023,676,3.0,54764,493.42
4,5,余杭区,48.5,1,1,1,15,18,1,1.0,...,C,11016.0,0.44,1.34,6.36,4/23/2023,727,1.0,30432,147.60


In [4]:
df.shape

(2500, 26)

This dataset has 26 features for this project We will use main 13 features


In [5]:
df.columns

Index(['id', 'district', 'area_sqm', 'bedrooms', 'living_rooms', 'bathrooms',
       'floor', 'total_floors', 'has_elevator', 'building_age_years',
       'orientation', 'decoration', 'property_type', 'developer',
       'subway_line', 'subway_distance_m', 'school_district_tier',
       'west_lake_distance_m', 'green_ratio', 'parking_ratio',
       'management_fee_yuan_per_sqm', 'listing_date', 'listing_duration_days',
       'price_adjustments_count', 'unit_price_yuan_per_sqm',
       'total_price_wan'],
      dtype='object')

In [6]:
features = df[['district', 'area_sqm', 'bedrooms', 'bathrooms', 'floor',
               'has_elevator', 'building_age_years', 'decoration',
               'subway_line', 'subway_distance_m', 'parking_ratio',
               'property_type', 'west_lake_distance_m']]

In [7]:
features.shape
target = df['total_price_wan']
target.head()

0    597.60
1    368.81
2     85.02
3    493.42
4    147.60
Name: total_price_wan, dtype: float64

In [8]:
features.values

array([['西湖区', 112.4, 3, ..., nan, '住宅 (Residential)', 24423.0],
       ['萧山区', 97.8, 3, ..., 0.64, '住宅 (Residential)', 1247.0],
       ['临安区', 70.4, 2, ..., 1.44, '住宅 (Residential)', 1146.0],
       ...,
       ['滨江区', 147.9, 4, ..., 1.28, '住宅 (Residential)', 7443.0],
       ['上城区', 108.2, 4, ..., nan, '住宅 (Residential)', nan],
       ['滨江区', 60.7, 1, ..., 1.22, '住宅 (Residential)', 13581.0]],
      shape=(2500, 13), dtype=object)

In [9]:
features.dtypes

district                 object
area_sqm                float64
bedrooms                  int64
bathrooms                 int64
floor                     int64
has_elevator              int64
building_age_years      float64
decoration               object
subway_line              object
subway_distance_m       float64
parking_ratio           float64
property_type            object
west_lake_distance_m    float64
dtype: object

In [10]:

obj=features.select_dtypes(include="object")
num=features.select_dtypes(exclude="object")

obj, num

(     district   decoration subway_line     property_type
 0         西湖区   毛坯 (Rough)      Line 2  住宅 (Residential)
 1         萧山区  豪华 (Luxury)      Line 2  住宅 (Residential)
 2         临安区   毛坯 (Rough)      Line 1  住宅 (Residential)
 3         西湖区  豪华 (Luxury)      Line 5    公寓 (Apartment)
 4         余杭区    精装 (Fine)      Line 2  住宅 (Residential)
 ...       ...          ...         ...               ...
 2495      拱墅区   毛坯 (Rough)     Line 16        别墅 (Villa)
 2496      钱塘区  豪华 (Luxury)     Line 10        别墅 (Villa)
 2497      滨江区    精装 (Fine)     Line 19  住宅 (Residential)
 2498      上城区  简装 (Simple)      Line 1  住宅 (Residential)
 2499      滨江区          NaN         NaN  住宅 (Residential)
 
 [2500 rows x 4 columns],
       area_sqm  bedrooms  bathrooms  floor  has_elevator  building_age_years  \
 0        112.4         3          3      8             1                 4.0   
 1         97.8         3          3      2             1                15.0   
 2         70.4         2        

In [11]:
print("Object Cols\n",obj.isnull().sum())
print("Numerical cols\n",num.isnull().sum())

Object Cols
 district           0
decoration       352
subway_line      417
property_type      0
dtype: int64
Numerical cols
 area_sqm                  0
bedrooms                  0
bathrooms                 0
floor                     0
has_elevator              0
building_age_years      243
subway_distance_m       248
parking_ratio           810
west_lake_distance_m    400
dtype: int64


In [12]:
from sklearn.impute import SimpleImputer

obj_imputer = SimpleImputer(strategy="most_frequent")
num_imputer = SimpleImputer(strategy="median")
obj=obj_imputer.fit_transform(obj)
num=num_imputer.fit_transform(num)

In [16]:
import numpy as np

X = np.concatenate([num, obj], axis=1)
y=df['total_price_wan']

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor

# -------------------------------------------------------
# 1. Load dataset
# -------------------------------------------------------
df = pd.read_csv("hangzhou_housing_dataset.csv")
print("DataFrame Imported")

# -------------------------------------------------------
# 2. Split features + target
# -------------------------------------------------------
target_column = "total_price_wan"     # <-- change if your target name is different
X = df.drop(columns=[target_column])
y = df[target_column]

# -------------------------------------------------------
# 3. Identify numeric & categorical columns
# -------------------------------------------------------
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "category"]).columns

# -------------------------------------------------------
# 4. Build preprocessing pipelines
# -------------------------------------------------------

# For numeric columns: impute missing with median
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# For categorical columns: impute most frequent + OneHotEncode
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

# -------------------------------------------------------
# 5. Build full model pipeline (Preprocess → XGBoost)
# -------------------------------------------------------
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("xgb", XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])

# -------------------------------------------------------
# 6. Train/test split
# -------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -------------------------------------------------------
# 7. Train the model
# -------------------------------------------------------
model.fit(X_train, y_train)

print("Training Completed")

DataFrame Imported
Training Completed


In [22]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MAE: 9.772820485687255
R²: 0.9910847822731833


In [23]:
import joblib
joblib.dump(model, "xgb_house_model.pkl")

['xgb_house_model.pkl']